In [ ]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti
import event_io as eio


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers
import pickle

# Some other useful packages 
import importlib
from pathlib import Path


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )
importlib.reload( eio )

Rdair=Co.Rdair()


In [ ]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [ ]:
%%time
importlib.reload( futi )
importlib.reload( nuti )
importlib.reload( auti )

nsteps=None
start_date=None
super_lat_range = [-90.,90.]

case, process_ncdata  = 'cam77_dyamond1_prod1'    , False

A = futi.read_case( case=case, nsteps=nsteps, start_date=start_date , super_lat_range=super_lat_range ) # , nsteps = 31*8 )

time, zlev, lat, lon = A.time, A.zlev, A.lat, A.lon

print( f"Got data into 'A' {A.keys()} ")

print( A.zeta.shape )

nt,nz,ny,nx = A.zeta.shape 

In [ ]:
%%time

cen_mark = 'tilt'
print( f"  'mark' for centroids = {cen_mark} " )

mark_x = np.transpose( A[cen_mark] , (2,3,0,1) )
pmid_x = np.transpose( A.pmid , (2,3,0,1) )


print( mark_x.shape )


mark_r = np.reshape( mark_x , ( ny*nx, nt, nz ) )
pmid_r = np.reshape( pmid_x , ( ny*nx, nt, nz ) )

ncol=ny*nx   #*nt
        
print( mark_r.shape )
print( A.pmid.shape )


plev=100_000. *np.exp( -A.zlev / 7_000. ) 
pmid=np.tile ( plev, ( ncol , 1) )

print(f"Now doing centroid calculation" )

k_steer = np.zeros( ( ncol, nt ) , dtype=int )
k_launch = np.zeros( ( ncol, nt ) , dtype=int )
c_steer = np.zeros( ( ncol, nt ) )
c_launch = np.zeros( (ncol,nt) )
for t in np.arange(nt):
    k_steer[:,t], k_launch[:,t], c_steer[:,t], c_launch[:,t] = auti.vorticity_centroid_levels(vorticity = mark_r[:,t,:], coord=pmid_r[:,t,:] , bound_surface=80_000., bound_top=2_000. )
    if (t%10==0):
        print( t )


k_steer  = np.transpose(  np.reshape( k_steer , (ny,nx,nt) )   , (2,0,1) )
k_launch = np.transpose(  np.reshape( k_launch , (ny,nx,nt) )   , (2,0,1) )



nt, nz, ny, nx = A.u.shape

precl  =  A.precl[:,-1,:,:]  #np.zeros( (nt,ny,nx) )

ks = k_steer.astype(int) # - 1
kl = k_launch.astype(int) # - 1

print(f"Now using centroids to extract fields for GWP" )

# epwp at launch level
epwp_zl = np.take_along_axis(
    A.rho_epwp, kl[:, None, :, :], axis=1 )[:, 0]

# tilt at launch level
tilt_zl = np.take_along_axis(
    A.tilt, kl[:, None, :, :], axis=1 )[:, 0]

# tilt at steering level
tilt_zs = np.take_along_axis(
    A.tilt, ks[:, None, :, :], axis=1 )[:, 0]

# tilt at steering level
zeta_zl = np.take_along_axis(
    A.zeta, kl[:, None, :, :], axis=1 )[:, 0]

# tilt at steering level
zeta_zs = np.take_along_axis(
    A.zeta, ks[:, None, :, :], axis=1 )[:, 0]

# U at steering level
u_zs = np.take_along_axis(
    A.u, ks[:, None, :, :], axis=1 )[:, 0]

# V at steering level
v_zs = np.take_along_axis(
    A.v, ks[:, None, :, :], axis=1 )[:, 0]

# U at launch level
u_zl = np.take_along_axis(
    A.u, kl[:, None, :, :], axis=1 )[:, 0]

# V at launch level
v_zl = np.take_along_axis(
    A.v, kl[:, None, :, :], axis=1 )[:, 0]

"""
# mean tilt over kl..ks (inclusive)
csum = np.concatenate(
    [np.zeros((nt, 1, ny, nx)), np.cumsum(A.tilt, axis=1)], axis=1 )

sum_hi = np.take_along_axis(csum, (ks + 1)[:, None, :, :], axis=1)[:, 0]
sum_lo = np.take_along_axis(csum,  kl[:, None, :, :],      axis=1)[:, 0]
sum_zs = np.take_along_axis(csum,  ks[:, None, :, :],      axis=1)[:, 0]

tilt_zs_zl = (sum_hi - sum_lo) / (ks - kl + 1)
tilt_zg_zs = (csum[:,nz,:,:] - sum_zs  ) / (nz - ks )
"""
tilt_delp = A.tilt*A.delp

csum_w = np.concatenate(
    [np.zeros((nt, 1, ny, nx)), np.cumsum(tilt_delp, axis=1, dtype=np.float64)], axis=1)
csum_p = np.concatenate(
    [np.zeros((nt, 1, ny, nx)), np.cumsum(A.delp, axis=1, dtype=np.float64)], axis=1)

# helper: gather at a (nt, ny, nx) index array
def _at(c, k):
    return np.take_along_axis(c, k[:, None, :, :], axis=1)[:, 0]

# kl..ks inclusive
num_zs_zl = _at(csum_w, ks + 1) - _at(csum_w, kl)
den_zs_zl = _at(csum_p, ks + 1) - _at(csum_p, kl)
tilt_zs_zl = num_zs_zl / den_zs_zl

# ks..nz-1 (steering level to surface)
num_zg_zs = csum_w[:, nz] - _at(csum_w, ks)
den_zg_zs = csum_p[:, nz] - _at(csum_p, ks)
tilt_zg_zs = num_zg_zs / den_zg_zs


filename=f"{case}_src_fit_vars_4.nc"

X = futi.write_src_fit_vars(filename, 
                   lat=A.lat, lon=A.lon, lev=A.zlev, nt=nt,
                   epwp_zl=epwp_zl,
                   tilt_zs_zl=tilt_zs_zl,
                   tilt_zg_zs=tilt_zg_zs,
                   tilt_zs=tilt_zs,
                   tilt_zl=tilt_zl,
                   zeta_zs=zeta_zs,
                   zeta_zl=zeta_zl,
                   u_zs=u_zs,
                   v_zs=v_zs,
                   u_zl=u_zl,
                   v_zl=v_zl,
                   k_launch=k_launch,
                   k_steer=k_steer,
                   precl=precl)


In [ ]:
tilt_delp = A.tilt*A.delp

In [ ]:
tilt_poo_zs_zl =np.zeros( (nt,ny,nx) )
tilt_poo_zg_zs =np.zeros( (nt,ny,nx) )

for t in np.arange( 1 ):
    for y in np.arange( ny ):
        for x in np.arange( nx ):
            ks_yx = k_steer[t,y,x]
            kl_yx = k_launch[t,y,x]
            tpoo = np.sum( tilt_delp[t,kl_yx:ks_yx+1,y,x] , axis=0 )
            ppoo = np.sum( A.delp[t,kl_yx:ks_yx+1,y,x] , axis=0 )
            tilt_poo_zs_zl[t,y,x] = tpoo/ppoo
            tpoo = np.sum( tilt_delp[t,ks_yx:nz,y,x] , axis=0 )
            ppoo = np.sum( A.delp[t,ks_yx:nz,y,x] , axis=0 )
            tilt_poo_zg_zs[t,y,x] = tpoo/ppoo
        #print( y )

            
        

In [ ]:

#plt.plot( tilt_zs_zl[0,40,:] )
plt.plot( tilt_zg_zs[0,40,:] )
plt.plot( tilt_poo_zg_zs[0,40,:] )
plt.plot( (tilt_zg_zs-tilt_poo_zg_zs)[0,40,:] )



In [ ]:

#plt.plot( tilt_zs_zl[0,40,:] )
plt.plot( tilt_zs_zl[0,40,:] )
plt.plot( tilt_poo_zs_zl[0,40,:] )
plt.plot( (tilt_zs_zl-tilt_poo_zs_zl)[0,40,:] )



In [ ]:
print( A.pint.shape )



delp

In [ ]:
%%time

case='xympas-DynZlZs-tilt-x02'
dates='2016-08-*'
foo=f"/glade/derecho/scratch/juliob/archive/GW_UnitTest/{case}/{case}.h.{dates}.nc"
Xgw=xr.open_mfdataset( foo , data_vars='different', coords='different', compat='no_conflicts' )


ny = Xgw.sizes["ny"]
nx = Xgw.sizes["nx"]

j = np.arange(ny).repeat(nx)
i = np.tile(np.arange(nx), ny)

X2 = (
    Xgw.assign_coords(_j=("ncol", j), _i=("ncol", i))     # mapping ncol -> (j,i)
     .set_index(ncol=("_j", "_i"))
     .unstack("ncol")
     .rename({"_j": "ny", "_i": "nx"})
     .assign_coords(ny=Xgw["lat_R"], nx=Xgw["lon_R"])        # put real coords on axes
)


X2['lat']=Xgw['lat_R']
X2['lon']=Xgw['lon_R']
Xgw=X2


lat1,lon1=Xgw.lat.values, Xgw.lon.values

k_steer_1  = Xgw.K_STEER_MOVMTN.values
k_launch_1 = Xgw.K_LAUNCH_MOVMTN.values
p_steer_1  = Xgw.P_STEER_MOVMTN.values
p_launch_1 = Xgw.P_LAUNCH_MOVMTN.values
pmid_mm_1 = Xgw.PMID_MOVMTN.values
pmid_1    = Xgw.PMID.values

xpwp_src_1_1  = Xgw.XPWP_SRC_1.values
xpwp_src_3_1  = Xgw.XPWP_SRC_3.values

tau_mm_1 = Xgw.TAU_MOVMTN.values
sgh=Xgw.SGH.values

xpwp_src_3_1[:,-1,:]=0.

z_steer_1 = -7_000. * np.log( p_steer_1 / 100_000. )
z_launch_1 = -7_000. * np.log( p_launch_1 / 100_000. )

usteer_1  = Xgw.USTEER_MOVMTN.values
ulaunch_1 = Xgw.ULAUNCH_MOVMTN.values

vsteer_1  = Xgw.VSTEER_MOVMTN.values
vlaunch_1 = Xgw.VLAUNCH_MOVMTN.values

uwavef_1 = Xgw.UWAVEF_MOVMTN.values
vwavef_1 = Xgw.VWAVEF_MOVMTN.values


In [ ]:
del_k_steer = k_steer - (k_steer_1 -1)

print( del_k_steer.shape )


#plt.plot( del_k_steer[0, 5:-5,:].flatten() ,'.')

hh = np.histogram( del_k_steer[:, 5:-5,:].flatten(), bins=np.linspace(-5,10,num=1001 ) )

plt.plot( hh[1][1:],hh[0] )


In [ ]:
u_zs3 = np.tile( u_zs[:,None,:,:] , (1,nz,1,1) )
v_zs3 = np.tile( v_zs[:,None,:,:] , (1,nz,1,1) )

print( u_zs3.shape )

uwavef = A.u - u_zs3 # np.zeros( (nt,nz,ny,nx) )
vwavef = A.v - v_zs3 # np.zeros( (nt,nz,ny,nx) )

In [ ]:
t,y,x = [100,40,100]

plt.plot( uwavef_1[t,:,y,x] )
plt.plot( uwavef[t,:,y,x] )




In [ ]:

u_wl = u_zl - u_zs
v_wl = v_zl - v_zs



In [ ]:
t,y,x = 100,40,12

fig,axs=plt.subplots( 1, 3, figsize=(22,6) )

ax=axs[0]
ax.plot( ulaunch_1[t,y,:] )
ax.plot( u_wl[t,y,:] )
ax.set_xlim( 10,40 )


ax=axs[1]
ax.plot( A.u[t,:,y,x] , zlev )
ax.plot( A.u[t, k_launch[t,y,x] ,y,x] , zlev[ k_launch[t,y,x] ], 'x'  , color='red' )
ax.plot( A.u[t, k_steer[t,y,x] ,y,x] , zlev[ k_steer[t,y,x] ], 'o'  , color='red' )
ax.plot( A.u[t, int(k_launch_1[t,y,x])-1 ,y,x]-0.5 , zlev[ int(k_launch_1[t,y,x])-1 ], 'x'  , color='green' )
ax.plot( A.u[t, int(k_steer_1[t,y,x])-1 ,y,x] -0.5 , zlev[ int(k_steer_1[t,y,x])-1 ], 'o'  , color='green' )

ax=axs[2]
ax.plot( uwavef[t,:,y,x] , zlev )
ax.plot( uwavef_1[t,:,y,x] , zlev )
ax.plot( uwavef[t, k_launch[t,y,x] ,y,x] , zlev[ k_launch[t,y,x] ], 'x'  , color='red' )
#ax.plot( A.u[t, k_steer[t,y,x] ,y,x] , zlev[ k_steer[t,y,x] ], 'o'  , color='red' )
#ax.plot( A.u[t, int(k_launch_1[t,y,x])-1 ,y,x] , zlev[ int(k_launch_1[t,y,x])-1 ], 'x'  , color='green' )
#ax.plot( A.u[t, int(k_steer_1[t,y,x])-1 ,y,x] , zlev[ int(k_steer_1[t,y,x])-1 ], 'o'  , color='green' )




In [ ]:
print( ulaunch_1.shape )
print( u_zl.shape )


t,y,x = [100,40,100]
print( usteer_1[t,y,x] )
print( u_zs[t,y,x] )

print( k_steer[t,y,x] )
print( k_steer_1[t,y,x] -1 )

print( A.u[t,  k_steer[t,y,x]-1 : k_steer[t,y,x]+2  ,y,x] )
plt.plot( A.u[t,:,y,x] )


In [ ]:
ulvs=2*np.linspace( -30,30,num=31 )
plt.contourf( usteer_1[10,:,:] ,levels=ulvs )

In [ ]:
plt.contourf( u_zs[10,:,:] ,levels=ulvs )

In [ ]:

print( precl.shape )
print( k_steer.shape )
print( k_launch.shape )
print( A.zlev.shape )


In [ ]:
Goo = xr.open_dataset( 'src_fit_vars_2.nc' )

In [ ]:
Goo

In [ ]:
%%time
poo= Goo.epwp_zl.values

In [ ]:
poo.shape

In [ ]:

#plt.scatter( epwp_zl.flatten() , np.tile( sgh , (nt,1,1) ).flatten() )
plt.scatter( epwp_zl.flatten() , precl.flatten() )



In [ ]:
t,y=90,10
plt.plot( tilt_zg_zs_1b[t,y,:] )
plt.plot( tilt_zg_zs_1[t,y,:] )


In [ ]:
nt,nz,ny,nx = np.shape( A.u )


z0=np.argmin( np.abs( zlev-0.))
z0p5=np.argmin( np.abs( zlev-500.))
z1=np.argmin( np.abs( zlev-1000.))
z3=np.argmin( np.abs( zlev-3000.))
z5=np.argmin( np.abs( zlev-5000.))
z6=np.argmin( np.abs( zlev-6000.))
z7=np.argmin( np.abs( zlev-7000.))
z8=np.argmin( np.abs( zlev-8000.))
z9=np.argmin( np.abs( zlev-9000.))

z10=np.argmin( np.abs( zlev-10000.))
z11=np.argmin( np.abs( zlev-11000.))
z12=np.argmin( np.abs( zlev-12000.))
z15=np.argmin( np.abs( zlev-15000.))
z17=np.argmin( np.abs( zlev-17000.))
z20=np.argmin( np.abs( zlev-20000.))
z22=np.argmin( np.abs( zlev-22000.))
z25=np.argmin( np.abs( zlev-25000.))


y0eq=np.argmin( np.abs( lat1-(0.) ))
y05n=np.argmin( np.abs( lat1-(5.) ))
y10n=np.argmin( np.abs( lat1-(10.) ))
y15n=np.argmin( np.abs( lat1-(15.) ))
y20n=np.argmin( np.abs( lat1-(20.) ))
y25n=np.argmin( np.abs( lat1-(25.) ))
y30n=np.argmin( np.abs( lat1-(30.) ))
y35n=np.argmin( np.abs( lat1-(35.) ))
y40n=np.argmin( np.abs( lat1-(40.) ))
y45n=np.argmin( np.abs( lat1-(45.) ))
y50n=np.argmin( np.abs( lat1-(50.) ))
y55n=np.argmin( np.abs( lat1-(55.) ))
y60n=np.argmin( np.abs( lat1-(60.) ))
y65n=np.argmin( np.abs( lat1-(65.) ))
y70n=np.argmin( np.abs( lat1-(70.) ))
y75n=np.argmin( np.abs( lat1-(75.) ))
y80n=np.argmin( np.abs( lat1-(80.) ))

y05s=np.argmin( np.abs( lat1-(-5.) ))
y10s=np.argmin( np.abs( lat1-(-10.) ))
y15s=np.argmin( np.abs( lat1-(-15.) ))
y20s=np.argmin( np.abs( lat1-(-20.) ))
y25s=np.argmin( np.abs( lat1-(-25.) ))
y30s=np.argmin( np.abs( lat1-(-30.) ))
y35s=np.argmin( np.abs( lat1-(-35.) ))
y40s=np.argmin( np.abs( lat1-(-40.) ))
y45s=np.argmin( np.abs( lat1-(-45.) ))
y50s=np.argmin( np.abs( lat1-(-50.) ))
y55s=np.argmin( np.abs( lat1-(-55.) ))
y60s=np.argmin( np.abs( lat1-(-60.) ))
y65s=np.argmin( np.abs( lat1-(-65.) ))
y70s=np.argmin( np.abs( lat1-(-70.) ))
y75s=np.argmin( np.abs( lat1-(-75.) ))
y80s=np.argmin( np.abs( lat1-(-80.) ))


print( zlev[30] )

In [ ]:

mask=sgh<10.

In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()

tau_mm = tau_mm_1
z_steer=z_steer_1
z_launch=z_launch_1



logmom=np.linspace( -5,1.,num=31)
t0,t1=100,100

y0,y1=y50s,y45s

fig,axs=plt.subplots( 1,3, figsize=(25,6) )

ax=axs[0]
c=ax.contourf(lon1, zlev, np.log10(  np.mean( tau_mm[t0:t1+1,1:,y0:y1+1,:], axis=(0,2) )  ),levels=logmom)

ax.plot( lon1,   np.mean( z_steer[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
ax.plot( lon1,   np.mean( z_launch[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
plt.colorbar(c)
#ax.set_ylim(-90,90)

ax=axs[1]
c=ax.contourf(lon1, zlev, np.log10(  np.mean( A.rho_epwp[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ) ,levels=logmom)
ax.plot( lon1,   np.mean( z_steer[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
ax.plot( lon1,   np.mean( z_launch[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
plt.colorbar(c)
#ax.set_ylim(-90,90)

ax=axs[2]
c=ax.contourf(lon1, zlev,  np.mean( A.rho_thpwp[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=np.linspace(-0.02,0.02,num=21),cmap='bwr')
ax.plot( lon1,   0+1000.*1e8*np.mean( A.precl[t0:t1+1,57,y0:y1+1,:], axis=(0,1) ) ) 
plt.colorbar(c)
#ax.set_ylim(-90,90)


In [ ]:
A['vmag'] = np.sqrt( A.u**2 + A.v**2 )

In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()

tau_mm = tau_mm_1
z_steer=z_steer_1
z_launch=z_launch_1


tiltlevs = 0.5e-6 * np.linspace( 0,1.,num=21)
vmaglevs = 50.*np.linspace( 0,1.,num=11)
logmom=np.linspace( -5,1.,num=31)
t0,t1=100,100
#t0,t1=0,nt-1

y0,y1=y50s,y45s
#y0,y1=y65s,y60s
y0,y1=y0eq,y0eq
y0,y1=y40s,y40s

fig,axs=plt.subplots( 1,4, figsize=(33,6) )

ax=axs[0]
c=ax.contourf(lon1, zlev, np.log10(  np.mean( tau_mm[t0:t1+1,1:,y0:y1+1,:], axis=(0,2) ) +1.e-8 ),levels=logmom)

ax.plot( lon1,   np.mean( z_steer[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
ax.plot( lon1,   np.mean( z_launch[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
plt.colorbar(c)
#ax.set_ylim(-90,90)

ax=axs[1]
c=ax.contourf(lon1, zlev, np.log10(  np.mean( A.rho_epwp[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ) ,levels=logmom)
ax.plot( lon1,   np.mean( z_steer[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
ax.plot( lon1,   np.mean( z_launch[t0:t1+1,y0:y1+1,:] , axis=(0,1) ) )
plt.colorbar(c)
#ax.set_ylim(-90,90)

ax=axs[2]
#c=ax.contourf(lon1, zlev,  np.mean( A.rho_thpwp[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=np.linspace(-0.02,0.02,num=21),cmap='bwr')
#c=ax.contourf(lon1, zlev,  np.mean( uwavef_2[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=21,cmap='bwr')
c=ax.contourf(lon1, zlev,  np.mean( A.tilt[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=tiltlevs,cmap='bwr')
l=ax.contour(lon1, zlev,  np.mean( A.vmag[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=vmaglevs )#,colors='black')
plt.colorbar(c)
#ax.set_ylim(-90,90)

ax=axs[3]
c=ax.contourf(lon1, zlev,  np.mean( A.rho_thpwp[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=np.linspace(-0.02,0.02,num=21),cmap='bwr')
ax.plot( lon1,   0+1000.*1e8*np.mean( (A.precc+A.precl)[t0:t1+1,57,y0:y1+1,:], axis=(0,1) ) ) 
#c=ax.contourf(lon1, zlev,  np.mean( uwavef_2[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=21,cmap='bwr')
#c=ax.contourf(lon1, zlev,  np.mean( A.u[t0:t1+1,:,y0:y1+1,:], axis=(0,2) )  ,levels=21,cmap='bwr')
plt.colorbar(c)
#ax.set_ylim(-90,90)


In [ ]:
A.precl.shape

In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()
mask=1.0 # sgh<1.


scale_prec=86_400.*1000.
logmom=np.linspace( -5,1.,num=31)
logmom=np.linspace( -4,0.,num=51)
t0,t1=0,nt-1
t0,t1=100,100
zp=z12
fig,axs=plt.subplots( 1,6, figsize=(49,6) )

cmap='gist_ncar'
n=0
ax=axs[n]
c=ax.contourf(lon1, lat1[6:], mask* np.mean( xpwp_src_3_1[t0:t1+1,6:,:],axis=0) ,levels=21)
#l=ax.contour(lon1, lat1, Xgw.SGH[t0:t1+1,:].values ,levels=[200,10000], colors='white')
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1[6:], mask* np.mean( xpwp_src_3_1[t0:t1+1,6:,:]**2,axis=0) ,levels=21)
#l=ax.contour(lon1, lat1, Xgw.SGH[t0:t1+1,:].values ,levels=[200,10000], colors='white')
plt.colorbar(c)
ax.set_ylim(-90,90)

"""
n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.mean( xpwp_src_3_2[t0:t1+1,:,:],axis=0) ,levels=21)
#l=ax.contour(lon1, lat1, Xgw.SGH[t0:t1+1,:].values ,levels=[200,10000], colors='white')
plt.colorbar(c)
ax.set_ylim(-90,90)
"""
n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10( np.mean( tau_mm_1[t0:t1+1,zp,:,:]+1e-8,axis=0)  ),levels=logmom-0, cmap=cmap)
plt.colorbar(c)
ax.set_ylim(-90,90)

"""
n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10( np.mean( tau_mm_2[t0:t1+1,zp,:,:]+1e-8,axis=0)  ),levels=logmom -0, cmap=cmap)
plt.colorbar(c)
ax.set_ylim(-90,90)
"""

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10( np.mean( A.rho_epwp[t0:t1+1,zp,:,:],axis=0)  ) ,levels=logmom, cmap=cmap)
#c=ax.contourf(lon1, lat1, mask* np.mean( A.precl[t0:t1+1,zp,:,:],axis=0)   ,levels=21, cmap=cmap)
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
#c=ax.contourf(lon1, lat1, mask*np.log10( np.mean( A.rho_epwp[t0:t1+1,zp,:,:],axis=0)  ) ,levels=logmom, cmap=cmap)
c=ax.contourf(lon1, lat1, mask* scale_prec* np.mean( (1*A.precl+1*A.precc)[t0:t1+1,zp,:,:],axis=0)   ,levels=np.linspace(0,20.,num=21  ), cmap=cmap)
plt.colorbar(c)
ax.set_ylim(-90,90)


In [ ]:
from scipy.ndimage import gaussian_filter

#smoothed = gaussian_filter(field, sigma=2)  # sigma in grid points

sgh_smooth = gaussian_filter( sgh, sigma=5) 



t0,t1=100,100
#t0,t1=0,nt-1
z0,z1=z9,z3
y0,y1=y40s,y25s
#y0,y1=y45s,y40s
#y0,y1=y0eq,y0eq

mask=sgh_smooth[y0:y1+1,:].flatten()<0.1

a,b=1,0.5
plt.scatter( mask*a * 1.e5*np.mean( A.tilt[ t0:t1+1,z0:z1,y0:y1+1,: ]**2 ,axis=1 ).flatten() + mask*b*(A.precl[t0:t1+1,z12,y0:y1+1,:].flatten()) , mask*A.rho_epwp[t0:t1+1,z12,y0:y1+1,:].flatten() )
#plt.scatter( np.mean( A.tilt[ t0:t1+1,z0:z1,y0:y1+1,: ],axis=1 ).flatten()      )

#plt.scatter( 0*A.precl[t0:t1+1,57,y0:y1+1,:] + 1*np.mean( A.tilt[ t0:t1+1,z0:z1,y0:y1+1,: ],axis=1 ) , A.rho_epwp[t0:t1+1,z12,y0:y1+1,:] )

In [ ]:
plt.plot( lon1,   1000.*1e8*np.mean( A.precl[t0:t1+1,57,y0:y1+1,:], axis=(0,1) ) ) 

In [ ]:

plt.plot( lon1, np.log10(  np.mean( tau_mm[t0:t1+1,z20,y0:y1+1,:]+1e-8, axis=(0,1) )  ) )
plt.plot( lon1, np.log10(  np.mean( A.rho_epwp[t0:t1+1,z20,y0:y1+1,:], axis=(0,1) )  ) )

In [ ]:
#plt.contourf( xpwp_src_3[0,:,:] )
#plt.colorbar()

mask=sgh<1.

logmom=np.linspace( -5,1.,num=31)
t=200

fig,axs=plt.subplots( 1,5, figsize=(41,6) )

n=0
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*xpwp_src_3_1[t,:,:] ,levels=21)
#l=ax.contour(lon1, lat1, Xgw.SGH[:,:].values ,levels=[200,10000], colors='white')
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*xpwp_src_3_2[t,:,:] ,levels=21)
#l=ax.contour(lon1, lat1, Xgw.SGH[:,:].values ,levels=[200,10000], colors='white')
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10(tau_mm_1[t,z15,:,:]+1e-5),levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10(tau_mm_2[t,z15,:,:]+1e-5),levels=21)
plt.colorbar(c)
ax.set_ylim(-90,90)

n=n+1
ax=axs[n]
c=ax.contourf(lon1, lat1, mask*np.log10(A.rho_epwp[t,z15,:,:]) ,levels=logmom)
plt.colorbar(c)
ax.set_ylim(-90,90)


In [ ]:
print(stop_this_thing)

In [ ]:
from scipy.ndimage import gaussian_filter

#smoothed = gaussian_filter(field, sigma=2)  # sigma in grid points

sgh_smooth = gaussian_filter( sgh, sigma=sigma) 
mask=sgh_smooth<.1



zeep=z15


nt,nz,ny,nx = np.shape( A.u )

tau_mm_1_smooth =np.zeros( (nt,ny,nx ) )
tau_mm_2_smooth =np.zeros( (nt,ny,nx ) )
repwp_smooth =np.zeros( (nt,ny,nx ) )
sigma=2

for t in np.arange (nt ):
    tmp=mask*tau_mm_1[t,zeep,:,:]
    tau_mm_1_smooth[t,:,:]  = gaussian_filter( tmp, sigma=sigma) 
    tmp=mask*tau_mm_2[t,zeep,:,:]
    tau_mm_2_smooth[t,:,:]  = gaussian_filter( tmp, sigma=sigma) 
    tmp=mask*A.rho_epwp[t,zeep,:,:]
    repwp_smooth[t,:,:]  = gaussian_filter( tmp, sigma=sigma) 
    

In [ ]:
from scipy import stats

print(y_pred_2.shape)
print(yv_2.shape)



eps = 1e-12
ly_pred_2       = np.log(np.maximum(y_pred_2,  eps))
ly_targ_2       = np.log(np.maximum(yv_2        ,  eps))
r_log, _ = stats.pearsonr(ly_targ_2, ly_pred_2)
print(r_log )

In [ ]:
from scipy import stats

#zlev=Eco.zlev

print(Eco.epwp_4D.shape)

#xv=Eco.zeta_4D[:,1,z6,4,:].mean(axis=1)
#xv=np.mean(Eco.zeta_4D[:,:,z10,4,:],axis=(1,2))
#xv=np.mean(Eco.zeta_4D[:,:,z7,:,:],axis=(1,2,3))
#xv=np.mean( Eco.tilt_4D[:,:,z7,:,:],axis=(1,2,3)  ) 

xvs=[]
xvs.append( np.mean( Eco.zeta_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.tilt_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
xvs.append( np.mean( Eco.fgf_4D[:,:,z10:z3,:,:],axis=(1,2,3,4)  ) )
flds=['zeta','tilt','fgf']

yv=np.mean( Eco.epwp_4D[:,:,z10,:,:],axis=(3,2,1) )
#yv=np.mean( Eco.epwp_4D[:,nt_v-1,z10,:,:],axis=(2,1) )
print(yv.shape)

fig,axs=plt.subplots( 1, len(xvs), figsize=( (len(xvs)*8, 4 ) ) )
ip=0
for xv in xvs:
    ax=axs[ip]
    ax.scatter( xv,yv )
    r, p = stats.pearsonr(xv, yv)
    print(f"Patch mean {flds[ip]} vs patch mean epwp: r={r:.3f}, p={p:.2e}")
    ip=ip+1
